# MEIKA Vox · transcribir audios desde tu Drive

**Gratis y de código abierto. No necesitas saber programar.**

1. En **Entorno de ejecución → Cambiar tipo de entorno**, selecciona **GPU** (T4 si aparece). La disponibilidad depende de Google; no necesitas contratar un plan para intentar usar la GPU gratuita.
2. Pulsa ▶ en **Preparar y abrir MEIKA Vox** y autoriza tu Drive. La primera instalación puede tardar varios minutos.
3. En el panel de esta misma celda, elige una carpeta o pulsa **Encontrar carpetas con audios**.
4. Elige cómo transcribir y pulsa **Transcribir N audios**. Al terminar podrás descargar los textos.

**Modalidades:** solo texto sin token; separar voces sin token (**experimental: calidad por validar**); separar voces con token gratuito de Hugging Face. La clave solo se configura para la tercera opción. Con GPU, las tres modalidades usan WhisperX large-v3 para el texto.


In [ ]:
#@title Preparar y abrir MEIKA Vox
import importlib.util
import subprocess
import sys
import threading
from pathlib import Path

from google.colab import drive, output
from IPython.display import clear_output

if any(t.name in {"MEIKA_Vox_batch", "MEIKA_Vox_search"} and t.is_alive()
       for t in threading.enumerate()):
    raise RuntimeError("Hay una tarea activa. Deténla desde el panel o espera a que termine.")

REPO = Path("/content/meika-vox")
# La interfaz y el motor se descargan de la misma versión pública.
REPO_REF = "fix/colab-engine-stability-20261009"


def checked(command):
    result = subprocess.run([str(item) for item in command], capture_output=True, text=True)
    if result.returncode:
        print(result.stderr[-3000:] or result.stdout[-3000:])
        raise RuntimeError(
            "La preparación falló. Comprueba la conexión y ejecuta esta celda de nuevo."
        )


print("1/3 · Preparando FFmpeg y Git…")
checked(["apt-get", "-qq", "update"])
checked(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"])
print("2/3 · Descargando MEIKA Vox…")
if not (REPO / ".git").exists():
    checked(["git", "clone", "--quiet", "https://github.com/MeikaLab/meika-vox.git", REPO])
checked(["git", "-C", REPO, "fetch", "--quiet", "origin", REPO_REF])
checked(["git", "-C", REPO, "checkout", "--quiet", "--detach", "FETCH_HEAD"])
print("3/3 · Instalando dependencias; la primera preparación puede tardar varios minutos…")
checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
         "-e", str(REPO) + "[whisperx]", "ipywidgets>=8,<9"])
# Si se reejecuta el cuaderno en la misma sesión, no conservar módulos de la versión anterior.
for name in list(sys.modules):
    if name == "meika_vox" or name.startswith("meika_vox."):
        del sys.modules[name]
sys.path.insert(0, str(REPO / "src"))
runtime = importlib.import_module("meika_vox.runtime").inspect_runtime()
if not runtime.asr_ready:
    raise RuntimeError(
        "La instalación quedó incompleta. Reinicia la sesión y ejecuta todo de nuevo."
    )
# Importar realmente los módulos en un proceso limpio: find_spec no detecta dependencias rotas.
probe = subprocess.run([sys.executable, "-c", """
import traceback
try:
    import torch, transformers, whisperx
    import whisperx.asr, whisperx.alignment
    from transformers import Pipeline
    print("Motor listo: imports PyTorch, Transformers y WhisperX correctos")
except Exception:
    traceback.print_exc()
    raise
"""], capture_output=True, text=True, timeout=180)
if probe.returncode:
    print(probe.stderr[-12000:] or probe.stdout[-12000:])
    raise RuntimeError("El motor no pudo importarse; revisa el diagnóstico anterior.")
print("Preparación comprobada. Ahora autoriza tu Google Drive.")

drive.mount("/content/drive")
output.enable_custom_widget_manager()
spec = importlib.util.spec_from_file_location("vox_ui", REPO / "scripts/colab_ui.py")
ui = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ui)
# El panel aparece en esta primera celda; se retiran los mensajes de preparación.
clear_output(wait=True)
_panel = ui.build_panel(REPO, Path("/content/drive/MyDrive"))


**Gratis y de código abierto.** Colab procesa los audios en Google y requiere acceso a tu Drive; no se publican en GitHub. GPU y duración variables. Revisa las transcripciones antes de usar citas.
